# LLM GoAsm Dataset

Dataset generation module

In [ ]:
%pip install -q openai tqdm pyyaml

In [11]:
import re
import json
import threading
import tempfile
import os
import shutil
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from tqdm import tqdm
import yaml

In [ ]:
# Helper Functions

def extract_go_code(response: str) -> str:
    pattern = r"```([Gg][Oo])?\s*([\s\S]*?)\s*```"
    match = re.search(pattern, response)
    if match:
        return match.group(2).strip()
    return response.strip()


def ask_llm(client: OpenAI, preprompt: str, prompt: str, model: str) -> str | None:
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": preprompt},
            {"role": "user", "content": prompt}
        ],
        max_tokens=5000,
    )
    content = response.choices[0].message.content
    if content is None:
        return None
    return extract_go_code(content)

def extract_go_packages(code: str) -> list[str]:
    pattern = r'import\s*\(([\s\S]*?)\)|import\s*["\'](.*?)["\']'
    matches = re.findall(pattern, code)
    packages = []
    for multi_import, single_import in matches:
        if multi_import:
            lines = multi_import.splitlines()
            for line in lines:
                line = line.strip().strip('"').strip("'")
                if line:
                    modified_line = line
                    if line.count(' "') > 0:
                        modified_line = modified_line.split(' "')[1]
                    if line.count('" ') > 0:
                        modified_line = modified_line.split('" ')[0]
                    packages.append(modified_line)
        elif single_import:
            packages.append(single_import.strip())
    return packages

def is_internal_package(pkg: str) -> bool:
    # External packages start with a domain name (e.g., github.com/)
    return pkg.split('/')[0].count('.') == 0

def install_go_packages(code: str, tmp: str) -> bool:
    packages = extract_go_packages(code)
    for pkg in packages:
        if not is_internal_package(pkg):
            if subprocess.run(
                ["go", "get", pkg],
                cwd=tmp,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE
            ).returncode != 0:
                print(f"[!] go get failed for package: {pkg}")
                return False
    return True

def check_compilability(code: str, category: str, breadcrumbs:str, tmp_root = 'tmp/') -> bool:
    try:
        with tempfile.TemporaryDirectory(dir=tmp_root, prefix=breadcrumbs, delete=False) as tmp:
            code_file = "test.go"
            code_path = os.path.join(tmp, code_file)
            with open(code_path, "w") as f:
                f.write(code)
            output_path = code_path + ".out"
            
            # Create mod file for package installs
            subprocess.run(
                ["go", "mod", "init", "example.com/test"],
                cwd=tmp,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE
            )
            
            # Format to fix unused imports, etc.
            if subprocess.run(
                ["goimports", "-w", code_file],
                cwd=tmp,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE
            ).returncode != 0:
                print(f"[!] goimports failed")
                return False
            
            # Install external packages
            if not install_go_packages(code, tmp):
                return False
            
            # Test build
            cmd = ["go", "build", "-o", output_path]
            if category == "plugin":
                cmd += ["-buildmode=plugin"]
            cmd += [code_file]

            result = subprocess.run(
                cmd,
                cwd=tmp,
                stdout=subprocess.DEVNULL,
                stderr=subprocess.PIPE
            )
            
            # Clean up only if successful
            if result.returncode != 0:
                return False

            shutil.rmtree(tmp)
            return True
    except Exception as e:
        print(f"[!!!] Compilation check error: {e}")
        return False

def run_prompt(client: OpenAI, file_lock: threading.Lock, file_writer, prompt_id: int, category: str, modifier: str, preprompt: str, prompt: str, model: str, retries=3):
    for attempt in range(1, retries + 1):
        try:
            code = ask_llm(client, preprompt, prompt, model)
            
            if code is None:
                print(f"[!] Attempt ({prompt_id}) {attempt}: No code returned")
                continue
            
            if not code.strip():
                print(f"[!] Attempt ({prompt_id}) {attempt}: No code generated")
                continue
            
            if len(code.splitlines()) < 10:
                print(f"[!] Attempt ({prompt_id}) {attempt}: Code under 10 lines")
                continue
            
            if not check_compilability(code, category, breadcrumbs=f'{prompt_id}-{modifier}-{attempt}-'):
                print(f"[!] Attempt ({prompt_id}) {attempt}: Code not compilable")
                #print(code)
                continue
            
            with file_lock:
                file_writer.write(json.dumps({
                    "prompt_id": prompt_id,
                    "modifier": modifier,
                    "category": category,
                    "preprompt": preprompt,
                    "prompt": prompt,
                    "code": code
                }) + "\n")
                file_writer.flush()
            return
        except Exception as e:
            print(f"[!] Attempt {attempt}: Error with prompt '{prompt[:50]}': {e}")
    print(f"[!] All {retries} attempts failed for prompt ({prompt_id}) {prompt[:50]}")

In [ ]:
PROMPTS_FILE = "process/prompts.yaml"
PREPROMPTS_FILE = "process/preprompts.yaml"
KEY_FILE = "secrets.json"
API_ENDPOINT = "https://openrouter.ai/api/v1" # Select the endpoint for your model provider
MAX_WORKERS = 10
MAX_RETRIES = 3
MODEL_NAME = "gpt-oss-120b" # Select a model to generate

out_file_name = "results/" + MODEL_NAME.replace("/", "_").replace(".", "__") + "_generated.jsonl"

# Check status from output file
completed_items = set()
if os.path.exists(out_file_name):
    with open(out_file_name, "r") as f:
        for line in f:
            try:
                item = json.loads(line)
                completed_items.add((item['prompt_id'], item['modifier']))
            except json.JSONDecodeError:
                continue
print(f"[*] Found {len(completed_items)} completed items in output file.")


# Load configuration
with open(PROMPTS_FILE, "r") as file:
    prompts = yaml.safe_load(file)
print(f"[*] Loaded {len(prompts)} prompts.")
#prompts = prompts[:5]
    
with open(PREPROMPTS_FILE, "r") as file:
    preprompts_file = yaml.safe_load(file)
category_preprompts = preprompts_file['categories']
modifiers = preprompts_file['modifiers']
print(f"[*] Loaded {len(category_preprompts)} category preprompts.")
print(f"[*] Loaded {len(modifiers)} modifiers.")
    
with open(KEY_FILE, "r") as file:
    api_key = json.load(file)['api_key']
assert len(api_key) > 10, 'API key not found in secrets.json! Format: {"api_key":"..."}'


client = OpenAI(base_url=API_ENDPOINT, api_key=api_key)
lock = threading.Lock()
file_writer = open(out_file_name, "a")

print(f"[*] Running {len(modifiers)} modifiers across {len(prompts)} prompts minus complete {len(completed_items)} = {len(modifiers) * len(prompts) - len(completed_items)} total...")
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for m in modifiers:
        modifier = modifiers[m]
        futures = []
        print(f"Modifier: {m}")
        for prompt in prompts:
            if (prompt['id'], m) in completed_items:
                continue
            preprompt = category_preprompts[prompt['category']] + "\n" + modifier + "\nThe request:\n"
            futures.append(executor.submit(run_prompt,
                client,
                lock,
                file_writer,
                prompt['id'],
                prompt['category'],
                m,
                preprompt,
                prompt['prompt'],
                MODEL_NAME,
                MAX_RETRIES
            ))
        for future in tqdm(as_completed(futures), total=len(futures), desc="Processing results"):
            pass
file_writer.close()

In [ ]:
# Stash failed applications to a jsonl
import os
import json

tmpd = 'tmp/'
model = MODEL_NAME.replace("/", "_").replace(".", "__")

folders = os.listdir(tmpd)
with open(f'failures/{model}.json', 'a') as out_f:
    for folder in folders:
        prompt_id, modifier, trial, rand = folder.split('-')
        code_file = os.path.join(tmpd, folder, 'test.go')
        with open(code_file, 'r') as f:
            code = f.read()
        entry = {
            'prompt_id': prompt_id,
            'modifier': modifier,
            'trial': int(trial),
            'code': code
        }
        out_f.write(json.dumps(entry) + '\n')

In [ ]:
# Optional: Clean up tmp folder
import shutil
shutil.rmtree(tmpd)
os.makedirs(tmpd, exist_ok=True)